In [ ]:
!pip install -q -U google-genai

In [ ]:
# ============================================================
# GEMINI S/P CLASSIFICATION
# ============================================================

!pip install -q google-genai scikit-learn openpyxl

# ------------------------------------------------------------
# 1. Connect Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:


# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------

import pandas as pd
import json

from google import genai
from google.colab import userdata

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ------------------------------------------------------------
# 3. File path
# ------------------------------------------------------------

FILE_PATH = "/content/drive/MyDrive/validation_test_200.xlsm"

OUTPUT_PATH = (
    "/content/drive/MyDrive/Gemini_SP_200_validationTest.xlsx"
)


# ------------------------------------------------------------
# 4. Load data
# ------------------------------------------------------------

df = pd.read_excel(FILE_PATH)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))


# Take all 200 posts
test_df = df.copy()

# Ground-truth labels
test_df["label"] = test_df["label"].astype(int)

print("\nAll 850 ground-truth distribution:")
print(test_df["label"].value_counts().sort_index())

Columns: ['Post_ID', 'Title', 'Body', 'URL', 'label']
Total rows: 200

All 850 ground-truth distribution:
label
0     66
1    134
Name: count, dtype: int64


In [ ]:
from google import genai

API_KEY = "Enter API key"

client = genai.Client(api_key=API_KEY)

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply only with: API is working"
)

print(response.text)

API is working


In [ ]:


# ------------------------------------------------------------
# 6. Classification Prompt
# ------------------------------------------------------------

PROMPT = '''

You are a cybersecurity and privacy classification expert analyzing Reddit
posts about OpenClaw, an autonomous AI-agent system.

TASK
For EACH post, predict whether the post meaningfully discusses SECURITY
or PRIVACY.

LABELS
1 = Security or Privacy related
0 = Not Security or Privacy related

CORE DECISION RULE
Assign 1 only when the post contains at least one CONCRETE security/privacy
concern, question, claim, incident, risk, data-handling practice, access-control
issue, attack/vulnerability, or protective security/privacy mechanism.

Do NOT assign 1 merely because a security/privacy-associated word appears.
The meaning of that word in context must actually concern security or privacy.

Silently apply the following test to every post:

STEP 1 — POSITIVE EVIDENCE
Ask whether the post meaningfully discusses at least one of these:

A. PRIVACY / CONFIDENTIALITY / DATA HANDLING
- Exposure or leakage of personal, confidential, sensitive, or private data
- Password, API key, credential, secret, or authentication-token exposure
- Collection, monitoring, telemetry, tracking, retention, deletion, or sharing
  of user data when privacy is the issue
- Third-party access to user information
- Local processing or self-hosting when explicitly motivated by privacy,
  confidentiality, or keeping data under the user's control

B. AUTHENTICATION / AUTHORIZATION / ACCESS CONTROL
- Authentication or identity verification
- Who is allowed to access an agent, account, service, file, tool, or system
- Authorization, allowlists, access restrictions, least privilege
- Excessive or unintended permissions
- Credential or secret management when protection/security is the concern
- Unauthorized access or use

C. ATTACKS / VULNERABILITIES / COMPROMISE
- Prompt injection or indirect prompt injection
- Security-relevant jailbreaking
- Malicious skills, plugins, extensions, packages, or supply-chain threats
- Malware
- Vulnerabilities or exploitation
- Privilege escalation
- Account, agent, host, or system compromise
- Remote code execution
- A malicious or unauthorized party causing the agent/system to reveal data
  or perform protected actions

D. SECURITY CONTROLS / PROTECTIVE MECHANISMS
- Sandboxing or isolation used to contain the agent or limit damage
- Encryption or secure storage
- Security monitoring, auditing, or threat detection
- Skill/plugin verification or scanning
- Security guardrails
- Security/privacy best practices

E. SECURE DEPLOYMENT / NETWORK EXPOSURE
- Secure deployment or hardening
- Publicly exposed services when exposure is treated as a security risk
- Firewall, VPN, tunnel, network segmentation, or remote access when used
  to protect access or reduce security exposure
- Secure configuration when the purpose is protection rather than ordinary setup

If at least one of A-E is concretely present, continue to STEP 2.
If none is present, assign 0.

STEP 2 — EXCLUSION / CONTEXT CHECK
Even if a keyword appears, assign 0 when the actual topic is only ordinary
functionality, setup, reliability, cost, or performance.

Important negative boundaries:

- API key / OAuth / token / authentication mentioned only as a normal way to
  connect a model, pay for a service, or complete ordinary configuration -> 0
  UNLESS the post discusses exposure, storage, misuse, unauthorized access,
  access restrictions, or protection of those credentials.

- "permission" mentioned only as a software flag or capability needed to make
  a feature work -> 0
  UNLESS the concern is excessive, unintended, unauthorized, or restricted access.

- "tracking", "monitoring", "logs", or "traces" used only for token usage,
  model cost, debugging, performance, observability, or workflow analytics -> 0
  UNLESS the post concerns surveillance, personal-data collection, security
  auditing, or sensitive information in logs.

- "safe", "risk", or "trust" referring only to reliability, financial loss,
  model quality, platform-ban risk, business uncertainty, or whether the agent
  will work correctly -> 0.

- "private" referring only to a private repo, private message, private channel,
  or ordinary personal context -> 0
  UNLESS the post is actually about confidentiality, exposure, sharing,
  access, retention, or protection of that information.

- self-hosting, local models, Docker, VM, VPS, cloud, Tailscale, VPN, proxy,
  network, or remote access mentioned only for cost, convenience, installation,
  connectivity, or performance -> 0
  UNLESS security/privacy protection or exposure is a meaningful concern.

- sandbox mentioned only as a product/test environment or service limitation -> 0
  UNLESS it is being used to isolate OpenClaw, restrict capabilities, or contain risk.

- general autonomy, agent control, reliability, or "can the agent do X?" -> 0
  UNLESS the issue is authorization, access restriction, unintended privileged
  action, compromise, or protection of sensitive resources.

- general installation errors, configuration errors, coding problems, UI issues,
  feature requests, model recommendations, hardware, cost, speed, workflow,
  productivity, or business discussions -> 0 unless a concrete security/privacy
  issue is also discussed.

CONTRASTIVE EXAMPLES
These examples illustrate the boundary. They are not keyword rules.

Example A:
"How do I add my OpenAI API key? It keeps saying authentication failed."
-> 0
Reason: ordinary configuration; no credential-security concern.

Example B:
"Where does OpenClaw store my API key, and can installed skills read it?"
-> 1
Reason: credential exposure and unauthorized access are concrete concerns.

Example C:
"I access my dashboard through Tailscale, but the connection keeps dropping."
-> 0
Reason: connectivity/reliability only.

Example D:
"Is it safe to expose the dashboard publicly, or should I restrict it with
Tailscale and a firewall?"
-> 1
Reason: public exposure and protected remote access are the topic.

Example E:
"I built a dashboard that tracks API usage, tokens, and model cost."
-> 0
Reason: operational tracking, not privacy/security monitoring.

Example F:
"Does OpenClaw track my conversations or send usage data to third parties?"
-> 1
Reason: privacy-related collection and third-party data sharing.

Example G:
"The agent needs file-write permission, but the coding task still fails."
-> 0
Reason: permission is only a functional/configuration detail.

Example H:
"How can I let OpenClaw read one folder but prevent it from reading the rest
of my files?"
-> 1
Reason: access restriction and least privilege.

TIE-BREAKER
- If security/privacy relevance depends mainly on guessing from a keyword or
  making an unsupported assumption, assign 0.
- If a concrete security/privacy issue is clearly described even without words
  like "security" or "privacy", assign 1.
- If a post contains both a normal technical topic and a meaningful concrete
  security/privacy issue, assign 1.

READING RULES
1. Read both TITLE and BODY.
2. Use the whole context, not isolated words.
3. Do not infer incidents or risks that the author did not describe.
4. Return exactly one label for every post.
5. Do not output explanations or reasoning.

OUTPUT
Return ONLY valid JSON in exactly this structure:

{
  "results": [
    {
      "post_number": 1,
      "predicted_label": 1
    },
    {
      "post_number": 2,
      "predicted_label": 0
    }
  ]
}

POSTS:

{posts}

'''


In [ ]:

# ------------------------------------------------------------
# 7. Prepare the 850 posts in batches
# ------------------------------------------------------------

BATCH_SIZE = 10 # Further reduced batch size for more reliable JSON responses

batched_prompts = []

for i in range(0, len(test_df), BATCH_SIZE):
    batch_df = test_df.iloc[i : i + BATCH_SIZE]
    current_posts = ""

    for post_number, (_, row) in enumerate(
        batch_df.iterrows(),
        start=i + 1  # Continue numbering from the overall dataset
    ):
        title = (
            str(row["Title"])
            if pd.notna(row["Title"])
            else ""
        )

        body = (
            str(row["Body"])
            if pd.notna(row["Body"])
            else ""
        )

        current_posts += f"""

POST {post_number}

TITLE:
{title}

BODY:
{body}

----------------------------

"""

    batched_prompts.append(PROMPT.replace("{posts}", current_posts))

print(f"Prepared {len(batched_prompts)} batches for processing.")


Prepared 20 batches for processing.


In [ ]:
import time
from pydantic import BaseModel


class PostResult(BaseModel):
  post_number: int
  predicted_label: int


class BatchResponse(BaseModel):
  results: list[PostResult]


all_results = []
MAX_RETRIES = 3

for i, prompt_batch in enumerate(batched_prompts):
  print(f"Processing batch {i + 1}/{len(batched_prompts)}...")

  batch_success = False

  for attempt in range(1, MAX_RETRIES + 1):
    try:
      response = client.models.generate_content(
          model="gemini-2.5-flash",
          contents=prompt_batch,
          config={
              "temperature": 0.0,
              "response_mime_type": "application/json",
              "response_schema": BatchResponse,
          },
      )

      # Use SDK's built-in parser instead of json.loads(response.text)
      if response.parsed and response.parsed.results:
        batch_items = [item.model_dump() for item in response.parsed.results]
        all_results.extend(batch_items)
        batch_success = True
        break

    except Exception as e:
      print(f"  Attempt {attempt} failed for batch {i + 1}: {e}")
      time.sleep(2)

  if not batch_success:
    raise RuntimeError(
        f"Batch {i + 1} failed consistently across {MAX_RETRIES} attempts."
    )

results = all_results
print(f"\nSuccessfully processed all {len(results)} predictions.")

Processing batch 1/20...
Processing batch 2/20...
Processing batch 3/20...
Processing batch 4/20...
Processing batch 5/20...
Processing batch 6/20...
Processing batch 7/20...
Processing batch 8/20...
Processing batch 9/20...
Processing batch 10/20...
Processing batch 11/20...
Processing batch 12/20...
Processing batch 13/20...
Processing batch 14/20...
Processing batch 15/20...
Processing batch 16/20...
Processing batch 17/20...
Processing batch 18/20...
Processing batch 19/20...
Processing batch 20/20...

Successfully processed all 200 predictions.


In [ ]:




# ------------------------------------------------------------
# 10. Check Gemini returned exactly the expected number of posts
# ------------------------------------------------------------

if len(results) != len(test_df):

    raise ValueError(
        f"Gemini returned {len(results)} predictions. "
        f"Expected exactly {len(test_df)}."
    )

In [ ]:


# ------------------------------------------------------------
# 11. Match post number to prediction
# ------------------------------------------------------------

prediction_dict = {}

for result in results:

    post_number = int(
        result["post_number"]
    )

    prediction = int(
        result["predicted_label"]
    )

    if prediction not in [0, 1]:

        raise ValueError(
            f"Invalid label for Post {post_number}: "
            f"{prediction}"
        )

    prediction_dict[post_number] = prediction


predictions = []

for i in range(1, len(test_df) + 1):

    if i not in prediction_dict:

        raise ValueError(
            f"No prediction returned for Post {i}"
        )

    predictions.append(
        prediction_dict[i]
    )


test_df["predicted_label"] = predictions


# ------------------------------------------------------------
# 12. Ground Truth vs Gemini
# ------------------------------------------------------------

y_true = test_df["label"]

y_pred = test_df["predicted_label"]


In [ ]:

# ------------------------------------------------------------
# 13. PERFORMANCE
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)


print("\n" + "=" * 55)

print(
    "GEMINI PERFORMANCE ON ALL 200 POSTS"
)

print("=" * 55)

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-score  : {f1 * 100:.2f}%"
)

print("=" * 55)



GEMINI PERFORMANCE ON ALL 200 POSTS
Accuracy  : 84.00%
Precision : 89.84%
Recall    : 85.82%
F1-score  : 87.79%


In [ ]:

# ------------------------------------------------------------
# 14. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print(cm)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


# ------------------------------------------------------------
# 15. Classification Report
# ------------------------------------------------------------

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "Not S/P",
            "S/P"
        ],
        digits=4,
        zero_division=0
    )
)




Confusion Matrix:
[[ 53  13]
 [ 19 115]]

TN: 53
FP: 13
FN: 19
TP: 115

Classification Report:
              precision    recall  f1-score   support

     Not S/P     0.7361    0.8030    0.7681        66
         S/P     0.8984    0.8582    0.8779       134

    accuracy                         0.8400       200
   macro avg     0.8173    0.8306    0.8230       200
weighted avg     0.8449    0.8400    0.8416       200



In [ ]:

# ------------------------------------------------------------
# 16. Correct / Incorrect
# ------------------------------------------------------------

test_df["correct"] = (
    test_df["label"]
    ==
    test_df["predicted_label"]
).astype(int)


# ------------------------------------------------------------
# 17. Display each prediction
# ------------------------------------------------------------

print("\nIndividual Predictions:")

display(
    test_df[
        [

            "Title"
            "label",
            "predicted_label",
            "correct"
        ]
    ]
)


# ------------------------------------------------------------
# 18. Save to Google Drive
# ------------------------------------------------------------

test_df.to_excel(
    OUTPUT_PATH, # Use the already defined OUTPUT_PATH variable
    index=False
)

print("\nSaved to:")
print(OUTPUT_PATH)



Individual Predictions:


,Post_ID,Title,label,predicted_label,correct
0,1r003ek,"[Update] You asked for a secure, ""always-on"" w...",1,1,1
1,1rzwo5c,How we got OpenClaw browser automations workin...,1,0,0
2,1sbledc,Best cheap api models for openclaw? for each s...,0,1,0
3,1rprenp,I signed a peace treaty with AI today… just in...,0,0,1
4,1r2u2i0,🏛️ LegionX — Server-Side AI Parliament 41 agen...,1,1,1
...,...,...,...,...,...
195,1s6lxm7,Question about session and heartbeat,1,0,0
196,1t9f1fe,I broke my AI agent setup constantly for month...,0,1,0
197,1rbs4cn,I built TitanClaw v1.0 in pure Rust in just on...,1,1,1
198,1qxd1po,Why costs can climb faster than expected,0,0,1



Saved to:
/content/drive/MyDrive/Colab Notebooks/openclaw/apply_classifier/Gemini_SP_200_validationTest.xlsx
